# Tahap 1: Membangun Case Base
## Alur Kerja
1. Inventarisasi PDF mentah di `data/pdf/`
2. Ekstraksi teks PDF → plain text (PyMuPDF, dengan fallback OCR untuk PDF hasil scan)
3. Pembersihan teks (hapus header/footer, watermark, normalisasi)
4. Validasi keutuhan teks (≥80% isi tersedia)
5. Simpan ke `data/raw/case_XXX.txt` + `logs/cleaning.log`


## 0. Setup & Instalasi Library

In [49]:
# Jalankan sekali saja jika library belum terinstall
# !pip install pymupdf pytesseract pdf2image tqdm rapidfuzz

# Catatan untuk OCR fallback (opsional, hanya dipakai jika ada PDF hasil scan):
# - pytesseract butuh Tesseract OCR terinstall di sistem:
#     Windows : https://github.com/UB-Mannheim/tesseract/wiki
#     Linux   : sudo apt install tesseract-ocr
#     Mac     : brew install tesseract
# - pdf2image butuh poppler:
#     Windows : https://github.com/oschwartz10612/poppler-windows/releases
#     Linux   : sudo apt install poppler-utils
#     Mac     : brew install poppler


In [ ]:
!pip install pymupdf pytesseract pdf2image tqdm rapidfuzz pandas

In [ ]:
import os
import re
import json
import logging
from pathlib import Path
from datetime import datetime

import fitz  
from tqdm import tqdm
from rapidfuzz import fuzz  

def _lazy_import_ocr():
    import pytesseract
    from pdf2image import convert_from_path
    return pytesseract, convert_from_path


## 1. Konfigurasi Path

Sesuaikan `BASE_DIR` jika struktur folder project Anda berbeda.


In [ ]:
BASE_DIR = Path(".").resolve().parent  

PDF_DIR = BASE_DIR / "data" / "pdf"
RAW_DIR = BASE_DIR / "data" / "raw"
LOG_DIR = BASE_DIR / "logs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"PDF source dir : {PDF_DIR}")
print(f"Raw output dir : {RAW_DIR}")
print(f"Log dir        : {LOG_DIR}")
print(f"PDF dir exists?  {PDF_DIR.exists()}")


PDF source dir : C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\pdf
Raw output dir : C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\raw
Log dir        : C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\logs
PDF dir exists?  True


In [ ]:
log_path = LOG_DIR / "cleaning.log"

logger = logging.getLogger("cbr_case_base")
logger.setLevel(logging.INFO)
logger.handlers.clear()

fh = logging.FileHandler(log_path, mode="w", encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

ch = logging.StreamHandler()
ch.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
logger.addHandler(ch)

logger.info("=== Mulai sesi pembersihan Case Base ===")


INFO | === Mulai sesi pembersihan Case Base ===


## 2. Inventarisasi Dokumen PDF

Memeriksa jumlah dan ukuran file PDF yang tersedia. Ketentuan tugas: **minimal 30 dokumen**.


In [53]:
pdf_files = sorted(PDF_DIR.glob("*.pdf"))

print(f"Jumlah PDF ditemukan: {len(pdf_files)}")
if len(pdf_files) < 30:
    print(f"⚠️  PERINGATAN: jumlah dokumen ({len(pdf_files)}) kurang dari syarat minimal 30!")
else:
    print("✅ Jumlah dokumen memenuhi syarat minimal (≥30).")

print("\nDaftar file beserta ukuran:")
for f in pdf_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    flag = "  ⚠️ besar, kemungkinan hasil scan" if size_mb > 5 else ""
    print(f"  {f.name:<60} {size_mb:>8.2f} MB{flag}")


Jumlah PDF ditemukan: 30
✅ Jumlah dokumen memenuhi syarat minimal (≥30).

Daftar file beserta ukuran:
  putusan_1002_pid.sus_2021_pn_jkt.tim_20260620230052.pdf          0.48 MB
  putusan_1037_pid.sus_2021_pn_jkt.tim_20260620225944.pdf          0.29 MB
  putusan_11_pid.sus_2022_pn_jkt.tim_20260620225936.pdf            0.48 MB
  putusan_202_pid.sus_2023_pn_jkt.tim_20260619115103.pdf           2.11 MB
  putusan_203_pid.sus_2023_pn_jkt.tim_20260620225510.pdf           1.64 MB
  putusan_212_pid.sus_2025_pn_jkt.tim_20260620224313.pdf           0.44 MB
  putusan_330_pid.sus_2025_pn_jkt.tim_20260620224225.pdf           0.33 MB
  putusan_351_pid.sus_2025_pn_jkt.tim_20260620223604.pdf           0.58 MB
  putusan_365_pid.sus_2023_pn_jkt.tim_20260620225531.pdf           0.79 MB
  putusan_463_pid.sus_2024_pn_jkt.tim_20260620225349.pdf          10.75 MB  ⚠️ besar, kemungkinan hasil scan
  putusan_464_pid.sus_2024_pn_jkt.tim_20260620224818.pdf           1.11 MB
  putusan_465_pid.sus_2024_pn_jkt.tim_2

## 3. Deteksi Jenis PDF (Teks Asli vs Hasil Scan)

PDF putusan MA RI umumnya berupa teks asli (selectable), namun beberapa bisa jadi hasil scan/gambar
(ditandai ukuran file besar tapi jumlah halaman sedikit, atau hasil ekstraksi teks kosong/sangat pendek).

> **Catatan (temuan dari debugging):** beberapa dokumen ternyata punya watermark dan blok
> disclaimer berupa **teks asli** (bisa diekstrak PyMuPDF), tetapi **badan konten putusan (dakwaan, identitas
> terdakwa, amar, dst.) berupa GAMBAR/raster** — bukan teks asli. Jika deteksi hanya mengecek "apakah ada teks
> sama sekali", dokumen seperti ini akan **lolos terdeteksi sebagai teks asli** (karena watermark/disclaimer-nya
> terbaca), padahal isi utamanya tidak akan terekstrak. Fungsi `is_scanned_pdf()` di bawah ini sudah diperbaiki
> untuk **menghapus boilerplate (watermark, disclaimer, header) dari hitungan dulu**, lalu mengukur sisa
> "konten asli" yang tersisa — pendekatan ini terbukti berhasil mendeteksi kasus tersembunyi semacam itu.


In [54]:
def is_scanned_pdf(pdf_path: Path, min_chars_per_page: int = 50, sample_pages: int = 3) -> bool:
    """
    Mendeteksi apakah PDF kemungkinan hasil scan (gambar) dengan mengecek jumlah KONTEN ASLI
    (bukan sekadar teks apa pun) yang dapat diekstrak langsung dari beberapa halaman sampel.

    PENTING: boilerplate (watermark, disclaimer, header berulang) dihapus dulu sebelum menghitung
    panjang teks, karena beberapa PDF punya watermark/disclaimer berupa teks asli namun badan
    konten putusannya sendiri berupa gambar/raster (hasil print-to-PDF/screenshot). Tanpa langkah
    ini, dokumen semacam itu akan keliru terdeteksi sebagai "bukan hasil scan".
    """
    try:
        doc = fitz.open(pdf_path)
        n_pages = len(doc)
        sample_idx = range(min(sample_pages, n_pages))

        total_content_chars = 0
        for i in sample_idx:
            raw_page_text = doc[i].get_text()
            # Hapus boilerplate dulu sebelum menghitung - lihat fungsi remove_boilerplate
            # di sel cleaning (didefinisikan ulang ringan di sini agar tidak bergantung urutan sel)
            content_only = re.sub(
                r"^(?:h?kama|ahkamah Agung Repub(?:lik(?: Indonesia)?)?|mah Agung Republik Indonesia|blik Indonesi)\s*$",
                "", raw_page_text, flags=re.IGNORECASE | re.MULTILINE,
            )
            content_only = re.sub(
                r"Disclaimer\s*\n?Kepaniteraan.*?Telp\s*:\s*021-384\s*3348\s*\(ext\.318\)",
                "", content_only, flags=re.IGNORECASE | re.DOTALL,
            )
            content_only = re.sub(
                r"(?:Halaman\s+\d+\s*(?:dari\s+\d+)?\s*)?Putusan Nomor\s*:?\s*\d+/Pid\.?\s*Sus/\d{4}/PN\.?\s*Jkt\.?\s*Tim\.?",
                "", content_only, flags=re.IGNORECASE,
            )
            content_only = re.sub(
                r"putusan\.mahkamahagung\.go\.id|Direktori Putusan Mahkamah Agung Republik Indonesia"
                r"|^\s*Halaman\s+\d+\s*$",
                "", content_only, flags=re.IGNORECASE | re.MULTILINE,
            )
            total_content_chars += len(content_only.strip())

        doc.close()
        avg_content_chars = total_content_chars / max(len(list(sample_idx)), 1)
        return avg_content_chars < min_chars_per_page
    except Exception as e:
        logger.warning(f"Gagal cek jenis PDF {pdf_path.name}: {e}")
        return False


In [ ]:
scan_status = {}
for f in tqdm(pdf_files, desc="Mendeteksi jenis PDF"):
    scan_status[f.name] = is_scanned_pdf(f)

n_scanned = sum(scan_status.values())
print(f"\nHasil deteksi:")
print(f"  PDF teks asli  : {len(pdf_files) - n_scanned}")
print(f"  PDF hasil scan : {n_scanned}")

if n_scanned > 0:
    print("\nFile yang terdeteksi sebagai hasil scan (akan diproses via OCR):")
    for name, is_scan in scan_status.items():
        if is_scan:
            print(f"  - {name}")


Mendeteksi jenis PDF: 100%|██████████| 30/30 [00:00<00:00, 105.02it/s]


Hasil deteksi:
  PDF teks asli  : 26
  PDF hasil scan : 4

File yang terdeteksi sebagai hasil scan (akan diproses via OCR):
  - putusan_463_pid.sus_2024_pn_jkt.tim_20260620225349.pdf
  - putusan_465_pid.sus_2024_pn_jkt.tim_20260620224836.pdf
  - putusan_524_pid.sus_2024_pn_jkt.tim_20260620225402.pdf
  - putusan_525_pid.sus_2024_pn_jkt.tim_20260620224652.pdf


## 4. Ekstraksi Teks PDF → Plain Text

- PDF teks asli  → diekstrak langsung dengan **PyMuPDF**.
- PDF hasil scan → diproses dengan **OCR (pytesseract + pdf2image)**.

Hasil ekstraksi mentah (belum dibersihkan) disimpan sementara di memori, baru dibersihkan di langkah berikutnya.


In [ ]:
def extract_text_pymupdf(pdf_path: Path) -> str:
    """Ekstraksi teks langsung menggunakan PyMuPDF."""
    doc = fitz.open(pdf_path)
    text_parts = [page.get_text() for page in doc]
    doc.close()
    return "\n".join(text_parts)


def extract_text_ocr(pdf_path: Path, dpi: int = 150, lang: str = "ind") -> str:
    """
    Ekstraksi teks menggunakan OCR untuk PDF hasil scan.

    PENTING - diproses PER HALAMAN (bukan convert_from_path() sekaligus untuk semua halaman),
    karena dokumen dengan banyak halaman (di dataset ini ditemukan ada yang 160-170 halaman)
    dapat menghabiskan memori jika seluruh halaman di-load sebagai gambar secara bersamaan.
    DPI diturunkan ke 150 (dari 200) sebagai trade-off wajar antara akurasi OCR dan penggunaan
    memori/waktu proses - cukup untuk dokumen teks standar yang disertai watermark sederhana.

    lang="ind" menggunakan model bahasa Indonesia Tesseract
    (pastikan sudah install: tesseract-ocr-ind / paket bahasa Indonesia).
    """
    pytesseract, convert_from_path = _lazy_import_ocr()

    doc = fitz.open(pdf_path)
    n_pages = len(doc)
    doc.close()

    text_parts = []
    for page_num in range(n_pages):
        images = convert_from_path(
            str(pdf_path), dpi=dpi, first_page=page_num + 1, last_page=page_num + 1
        )
        page_text = pytesseract.image_to_string(images[0], lang=lang)
        text_parts.append(page_text)
        del images  

    return "\n".join(text_parts)


def extract_text(pdf_path: Path, force_ocr: bool = False) -> tuple[str, str]:
    """
    Ekstrak teks dari PDF dengan strategi:
    1. Coba PyMuPDF dulu (cepat).
    2. Jika hasil terlalu pendek / force_ocr=True, fallback ke OCR.

    Returns: (text, method_used)
    """
    if not force_ocr:
        text = extract_text_pymupdf(pdf_path)
        if len(text.strip()) >= 200:  
            return text, "pymupdf"

    try:
        text = extract_text_ocr(pdf_path)
        return text, "ocr"
    except Exception as e:
        logger.error(f"OCR gagal untuk {pdf_path.name}: {e}")
        return "", "failed"


In [ ]:
raw_extracted = {}  

for f in tqdm(pdf_files, desc="Ekstraksi teks"):
    force_ocr = scan_status.get(f.name, False)
    text, method = extract_text(f, force_ocr=force_ocr)
    raw_extracted[f.name] = (text, method)

    status = "OK" if len(text.strip()) > 0 else "GAGAL/KOSONG"
    logger.info(f"Ekstraksi {f.name} | metode={method} | panjang={len(text)} karakter | status={status}")

print("\nRingkasan metode ekstraksi:")
methods_count = {}
for _, method in raw_extracted.values():
    methods_count[method] = methods_count.get(method, 0) + 1
for m, c in methods_count.items():
    print(f"  {m}: {c} file")


Ekstraksi teks:   3%|▎         | 1/30 [00:00<00:03,  8.40it/s]INFO | Ekstraksi putusan_1037_pid.sus_2021_pn_jkt.tim_20260620225944.pdf | metode=pymupdf | panjang=53412 karakter | status=OK
INFO | Ekstraksi putusan_11_pid.sus_2022_pn_jkt.tim_20260620225936.pdf | metode=pymupdf | panjang=157677 karakter | status=OK
Ekstraksi teks:  10%|█         | 3/30 [00:00<00:02, 12.82it/s]INFO | Ekstraksi putusan_202_pid.sus_2023_pn_jkt.tim_20260619115103.pdf | metode=pymupdf | panjang=764173 karakter | status=OK
INFO | Ekstraksi putusan_203_pid.sus_2023_pn_jkt.tim_20260620225510.pdf | metode=pymupdf | panjang=786835 karakter | status=OK
Ekstraksi teks:  17%|█▋        | 5/30 [00:00<00:05,  4.83it/s]INFO | Ekstraksi putusan_212_pid.sus_2025_pn_jkt.tim_20260620224313.pdf | metode=pymupdf | panjang=123841 karakter | status=OK
INFO | Ekstraksi putusan_330_pid.sus_2025_pn_jkt.tim_20260620224225.pdf | metode=pymupdf | panjang=84250 karakter | status=OK
Ekstraksi teks:  23%|██▎       | 7/30 [00:01<00:03,  6


Ringkasan metode ekstraksi:
  pymupdf: 26 file
  ocr: 4 file


## 5. Pembersihan Teks (Cleaning)

Langkah pembersihan sesuai ketentuan tugas:
1. Hapus header/footer berulang, nomor halaman, watermark.
2. Normalisasi spasi berlebih dan karakter aneh hasil OCR.
3. (Opsional) lower-case & hapus tanda baca — disimpan terpisah agar teks asli tetap ada untuk pembacaan manusia, sedangkan versi ternormalisasi dipakai untuk tahap berikutnya jika diperlukan.

> **Catatan verifikasi:** pola boilerplate di bawah ini sudah diuji langsung terhadap hasil ekstraksi PDF asli
> putusan PN Jakarta Timur (bukan asumsi generik), dan terbukti membersihkan watermark/header/disclaimer
> tanpa merusak konten putusan. Tetap disarankan mengecek 1-2 hasil `.txt` lain secara manual setelah pipeline
> dijalankan untuk seluruh 30 dokumen, karena beberapa putusan (terutama yang diproses lewat OCR) bisa punya
> variasi format kecil.


In [ ]:
WATERMARK_FRAGMENTS_RE = re.compile(
    r"^(?:h?kama|ahkamah Agung Repub(?:lik(?: Indonesia)?)?|mah Agung Republik Indonesia|blik Indonesi)\s*$",
    flags=re.IGNORECASE | re.MULTILINE,
)

HEADER_NOMOR_RE = re.compile(
    r"(?:Halaman\s+\d+\s*(?:dari\s+\d+)?\s*)?"             
    r"Putusan Nomor\s*:?\s*"                                    
    r"\d+/Pid\.?\s*Sus/\d{4}/PN\.?\s*Jkt\.?\s*Tim\.?",          
    flags=re.IGNORECASE,
)

DISCLAIMER_REFERENCE_SENTENCES = [
    "Kepaniteraan Mahkamah Agung Republik Indonesia berusaha untuk selalu mencantumkan informasi paling kini dan akurat",
    "sebagai bentuk komitmen Mahkamah Agung untuk pelayanan publik, transparansi dan akuntabilitas pelaksanaan fungsi peradilan",
    "Namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi",
    "Dalam hal Anda menemukan inakurasi informasi yang termuat pada situs ini atau informasi yang seharusnya ada",
    "namun belum tersedia, maka harap segera hubungi Kepaniteraan Mahkamah Agung RI melalui",
    "Email kepaniteraan mahkamahagung go id Telp 021-384 3348 ext 318",
]
DISCLAIMER_FUZZY_THRESHOLD = 68  


def is_disclaimer_line(line: str) -> bool:
    """Cek apakah satu baris kemungkinan bagian dari blok disclaimer Kepaniteraan,
    toleran terhadap typo hasil OCR, menggunakan fuzzy partial-ratio per kalimat referensi."""
    line_clean = line.strip()
    if len(line_clean) < 15:  
        return False
    line_lower = line_clean.lower()
    for ref in DISCLAIMER_REFERENCE_SENTENCES:
        if fuzz.partial_ratio(ref.lower(), line_lower) >= DISCLAIMER_FUZZY_THRESHOLD:
            return True
    return False


def remove_disclaimer_fuzzy(text: str) -> str:
    """Hapus semua baris yang terdeteksi sebagai bagian blok disclaimer (lihat is_disclaimer_line)."""
    lines = text.split("\n")
    kept = [line for line in lines if not is_disclaimer_line(line)]
    return "\n".join(kept)



PAGE_NUM_RE = re.compile(r"^\s*Halaman\s+\d+\s*$", flags=re.IGNORECASE | re.MULTILINE)


EXTRA_PATTERNS = [
    r"putusan\.mahkamahagung\.go\.id",                     
    r"Direktori Putusan Mahkamah Agung Republik Indonesia",
    r"Halaman\s+\d+\s+dari\s+\d+",                          
]
EXTRA_RE = re.compile("|".join(EXTRA_PATTERNS), flags=re.IGNORECASE | re.MULTILINE)


DISCLAIMER_STANDALONE_RE = re.compile(r"^\s*Disclaimer\s*$", flags=re.IGNORECASE | re.MULTILINE)


def remove_boilerplate(text: str) -> str:
    """Hapus header/footer/watermark berdasarkan pola dokumen MA RI PN Jakarta Timur."""
    text = WATERMARK_FRAGMENTS_RE.sub("", text)
    text = HEADER_NOMOR_RE.sub("", text)
    text = remove_disclaimer_fuzzy(text)        
    text = DISCLAIMER_STANDALONE_RE.sub("", text)  
    text = PAGE_NUM_RE.sub("", text)
    text = EXTRA_RE.sub("", text)
    return text


def normalize_whitespace(text: str) -> str:
    """Normalisasi spasi berlebih dan baris kosong berulang."""
    text = re.sub(r"[ \t]+", " ", text)          
    text = re.sub(r"\n{3,}", "\n\n", text)        
    text = re.sub(r" +\n", "\n", text)            
    return text.strip()


def clean_text(raw_text: str) -> str:
    """Pipeline pembersihan lengkap untuk satu dokumen."""
    text = remove_boilerplate(raw_text)
    text = normalize_whitespace(text)
    return text


In [ ]:
cleaned_texts = {}

for fname, (raw_text, method) in raw_extracted.items():
    if not raw_text.strip():
        cleaned_texts[fname] = ""
        continue
    cleaned_texts[fname] = clean_text(raw_text)

sample_name = pdf_files[0].name
print(f"=== Contoh: {sample_name} ===")
print("\n--- SEBELUM cleaning (300 karakter pertama) ---")
print(raw_extracted[sample_name][0][:300])
print("\n--- SESUDAH cleaning (300 karakter pertama) ---")
print(cleaned_texts[sample_name][:300])


=== Contoh: putusan_1002_pid.sus_2021_pn_jkt.tim_20260620230052.pdf ===

--- SEBELUM cleaning (300 karakter pertama) ---
hkama
ahkamah Agung Repub
ahkamah Agung Republik Indonesia
mah Agung Republik Indonesia
blik Indonesi
Direktori Putusan Mahkamah Agung Republik Indonesia
putusan.mahkamahagung.go.id
Halaman 1 dari 67 Putusan Nomor: 1002/Pid.Sus/2021/PN Jkt.Tim 
 
P U T U S A N 
Nomor: 1002/Pid.Sus/2022/PN Jkt.Tim 
 

--- SESUDAH cleaning (300 karakter pertama) ---
P U T U S A N
Nomor: 1002/Pid.Sus/2022/PN Jkt.Tim

”DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA”



Pengadilan Negeri Jakarta Timur yang mengadili perkara pidana dengan
acara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai
berikut dalam perkara Terdakwa:
Terdakwa ditangkap 


## 6. Validasi Keutuhan Teks

Ketentuan tugas: **periksa keutuhan teks, minimal 80% isi putusan tersedia**.

> **Catatan metodologi:** rasio panjang teks "sesudah / sebelum cleaning" **bukan** ukuran yang baik untuk
> keutuhan, karena boilerplate (watermark, header, disclaimer) berulang di **setiap halaman** — dokumen yang
> panjang (banyak halaman) akan punya proporsi boilerplate lebih besar, sehingga rasio retensinya rendah
> meskipun cleaning-nya sempurna. Membandingkan terhadap teks mentah hanya mengukur "berapa banyak boilerplate
> yang terhapus", bukan "apakah isi putusan masih lengkap".
>
> Pendekatan yang dipakai di sini menilai keutuhan dari dua sisi:
> 1. **Estimasi residu boilerplate** — hitung berapa baris watermark/header/disclaimer yang *masih tersisa*
>    setelah cleaning (seharusnya mendekati 0). Jika masih banyak, berarti regex belum menangkap semua variasi.
> 2. **Kelengkapan struktur dokumen** — cek keberadaan elemen kunci yang wajib ada di setiap putusan
>    (`PUTUSAN`, identitas Terdakwa, `MENGADILI`, amar putusan) dan panjang teks absolut yang wajar
>    (bukan dokumen kosong/terpotong).
>
> Ketiga indikator di atas digabung menjadi satu **skor persentase kelengkapan (0-100%)** per dokumen,
> supaya mudah dibaca/dilaporkan (sesuai permintaan ketentuan tugas "minimal 80% isi putusan tersedia").
> Skor ini BUKAN rasio panjang teks mentah vs bersih, melainkan kombinasi berbobot: struktur dokumen (60%), kebersihan dari boilerplate (25%), dan panjang wajar (15%).


In [ ]:
RESIDUAL_BOILERPLATE_RE = re.compile(
    r"disclaimer|kepaniteraan@mahkamahagung|putusan\.mahkamahagung\.go\.id"
    r"|^\s*Halaman\s+\d+\s*$",
    flags=re.IGNORECASE | re.MULTILINE,
)

REQUIRED_STRUCTURE_PATTERNS = {
    "header_putusan": r"P\s*U\s*T\s*U\s*S\s*A\s*N",
    "identitas_terdakwa": r"Terdakwa",
    "amar_mengadili": r"M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I",
}

MIN_ABSOLUTE_CHARS = 1000  
MAX_RESIDUAL_FOR_SCORE = 10 

WEIGHT_STRUCTURE = 60
WEIGHT_CLEANLINESS = 25
WEIGHT_LENGTH = 15


def compute_completeness_score(n_structure_ok: int, n_structure_total: int,
                                 residual_count: int, char_count: int) -> float:
    """
    Hitung skor kelengkapan 0-100% dari kombinasi berbobot 3 komponen:
    - Kelengkapan struktur dokumen (60%): proporsi elemen wajib yang ditemukan
    - Kebersihan boilerplate (25%): semakin banyak residu, semakin rendah skor
    - Panjang teks wajar (15%): skor penuh jika >= MIN_ABSOLUTE_CHARS, menurun proporsional jika kurang
    """
    score_structure = (n_structure_ok / n_structure_total) * WEIGHT_STRUCTURE

    cleanliness_ratio = max(0.0, 1 - (residual_count / MAX_RESIDUAL_FOR_SCORE))
    score_cleanliness = cleanliness_ratio * WEIGHT_CLEANLINESS

    length_ratio = min(1.0, char_count / MIN_ABSOLUTE_CHARS)
    score_length = length_ratio * WEIGHT_LENGTH

    total_score = score_structure + score_cleanliness + score_length
    return round(total_score, 1)


def validate_document(cleaned_text: str) -> dict:
    """
    Validasi keutuhan dokumen berdasarkan:
    1. Residu boilerplate yang tersisa (idealnya 0)
    2. Kelengkapan struktur wajib putusan
    3. Panjang teks absolut minimal
    Menghasilkan juga skor persentase kelengkapan gabungan (completeness_pct).
    """
    residual_count = len(RESIDUAL_BOILERPLATE_RE.findall(cleaned_text))

    structure_found = {
        name: bool(re.search(pattern, cleaned_text, flags=re.IGNORECASE))
        for name, pattern in REQUIRED_STRUCTURE_PATTERNS.items()
    }
    n_structure_ok = sum(structure_found.values())
    n_structure_total = len(REQUIRED_STRUCTURE_PATTERNS)

    is_long_enough = len(cleaned_text) >= MIN_ABSOLUTE_CHARS

    completeness_pct = compute_completeness_score(
        n_structure_ok, n_structure_total, residual_count, len(cleaned_text)
    )

    if residual_count > 5:
        status = "PERLU CEK MANUAL - residu boilerplate tinggi"
    elif n_structure_ok < n_structure_total:
        missing = [k for k, v in structure_found.items() if not v]
        missing_str = ", ".join(missing)
        status = f"PERLU CEK MANUAL - struktur tidak lengkap ({missing_str})"
    elif not is_long_enough:
        status = "PERLU CEK MANUAL - teks terlalu pendek"
    else:
        status = "OK"

    return {
        "residual_boilerplate": residual_count,
        "structure_complete": f"{n_structure_ok}/{n_structure_total}",
        "char_count": len(cleaned_text),
        "completeness_pct": completeness_pct,
        "status": status,
    }


In [61]:
validation_results = []

for fname in [f.name for f in pdf_files]:
    method = raw_extracted[fname][1]
    raw_len = len(raw_extracted[fname][0])
    cleaned = cleaned_texts[fname]

    if raw_len == 0 or not cleaned.strip():
        result = {
            "residual_boilerplate": -1,
            "structure_complete": "0/3",
            "char_count": 0,
            "completeness_pct": 0.0,
            "status": "GAGAL - teks kosong",
        }
    else:
        result = validate_document(cleaned)

    result["filename"] = fname
    result["method"] = method
    result["retain_ratio_info_only"] = round(len(cleaned) / raw_len, 3) if raw_len else 0.0
    validation_results.append(result)

    logger.info(
        f"Validasi {fname} | metode={method} | residu={result['residual_boilerplate']} | "
        f"struktur={result['structure_complete']} | kelengkapan={result['completeness_pct']:.1f}% | "
        f"status={result['status']}"
    )

import pandas as pd
val_df = pd.DataFrame(validation_results)
cols_order = ["filename", "method", "char_count", "residual_boilerplate",
              "structure_complete", "completeness_pct", "retain_ratio_info_only", "status"]
val_df = val_df[cols_order]
val_df


INFO | Validasi putusan_1002_pid.sus_2021_pn_jkt.tim_20260620230052.pdf | metode=pymupdf | residu=0 | struktur=3/3 | kelengkapan=100.0% | status=OK
INFO | Validasi putusan_1037_pid.sus_2021_pn_jkt.tim_20260620225944.pdf | metode=pymupdf | residu=0 | struktur=3/3 | kelengkapan=100.0% | status=OK
INFO | Validasi putusan_11_pid.sus_2022_pn_jkt.tim_20260620225936.pdf | metode=pymupdf | residu=0 | struktur=3/3 | kelengkapan=100.0% | status=OK
INFO | Validasi putusan_202_pid.sus_2023_pn_jkt.tim_20260619115103.pdf | metode=pymupdf | residu=1 | struktur=3/3 | kelengkapan=97.5% | status=OK
INFO | Validasi putusan_203_pid.sus_2023_pn_jkt.tim_20260620225510.pdf | metode=pymupdf | residu=1 | struktur=3/3 | kelengkapan=97.5% | status=OK
INFO | Validasi putusan_212_pid.sus_2025_pn_jkt.tim_20260620224313.pdf | metode=pymupdf | residu=0 | struktur=3/3 | kelengkapan=100.0% | status=OK
INFO | Validasi putusan_330_pid.sus_2025_pn_jkt.tim_20260620224225.pdf | metode=pymupdf | residu=0 | struktur=3/3 | kel

,filename,method,char_count,residual_boilerplate,structure_complete,completeness_pct,retain_ratio_info_only,status
0,putusan_1002_pid.sus_2021_pn_jkt.tim_202606202...,pymupdf,146723,0,3/3,100.0,0.680,OK
1,putusan_1037_pid.sus_2021_pn_jkt.tim_202606202...,pymupdf,35683,0,3/3,100.0,0.668,OK
2,putusan_11_pid.sus_2022_pn_jkt.tim_20260620225...,pymupdf,104992,0,3/3,100.0,0.666,OK
3,putusan_202_pid.sus_2023_pn_jkt.tim_2026061911...,pymupdf,497686,1,3/3,97.5,0.651,OK
4,putusan_203_pid.sus_2023_pn_jkt.tim_2026062022...,pymupdf,514618,1,3/3,97.5,0.654,OK
5,putusan_212_pid.sus_2025_pn_jkt.tim_2026062022...,pymupdf,83039,0,3/3,100.0,0.671,OK
6,putusan_330_pid.sus_2025_pn_jkt.tim_2026062022...,pymupdf,56476,0,3/3,100.0,0.670,OK
7,putusan_351_pid.sus_2025_pn_jkt.tim_2026062022...,pymupdf,198620,0,3/3,100.0,0.681,OK
8,putusan_365_pid.sus_2023_pn_jkt.tim_2026062022...,pymupdf,166510,0,3/3,100.0,0.682,OK
9,putusan_463_pid.sus_2024_pn_jkt.tim_2026062022...,ocr,351847,0,3/3,100.0,0.743,OK


In [62]:
n_ok = (val_df["status"] == "OK").sum()
n_warn = val_df["status"].str.startswith("PERLU CEK MANUAL").sum()
n_fail = (val_df["status"] == "GAGAL - teks kosong").sum()

print(f"Dokumen OK              : {n_ok}")
print(f"Perlu cek manual        : {n_warn}")
print(f"Gagal (teks kosong)     : {n_fail}")

if n_warn > 0 or n_fail > 0:
    print("\n⚠️  Dokumen yang perlu perhatian:")
    display_cols = ["filename", "method", "char_count", "completeness_pct", "status"]
    print(val_df[val_df["status"] != "OK"][display_cols].to_string(index=False))

print("\nCatatan: kolom 'retain_ratio_info_only' hanya untuk referensi (rasio panjang teks sebelum/sesudah")
print("cleaning) — TIDAK dipakai sebagai kriteria validasi karena bias terhadap dokumen panjang.")

print(f"\n=== Ringkasan Skor Kelengkapan (completeness_pct) ===")
print(f"Rata-rata        : {val_df['completeness_pct'].mean():.1f}%")
print(f"Minimum          : {val_df['completeness_pct'].min():.1f}%")
print(f"Maksimum         : {val_df['completeness_pct'].max():.1f}%")

below_threshold = val_df[val_df["completeness_pct"] < 80]
if len(below_threshold) > 0:
    print(f"\n⚠️  {len(below_threshold)} dokumen dengan kelengkapan < 80% (ketentuan tugas):")
    print(below_threshold[["filename", "completeness_pct", "status"]].to_string(index=False))
else:
    print(f"\n✅ Semua dokumen memenuhi syarat kelengkapan ≥80% sesuai ketentuan tugas.")


Dokumen OK              : 30
Perlu cek manual        : 0
Gagal (teks kosong)     : 0

Catatan: kolom 'retain_ratio_info_only' hanya untuk referensi (rasio panjang teks sebelum/sesudah
cleaning) — TIDAK dipakai sebagai kriteria validasi karena bias terhadap dokumen panjang.

=== Ringkasan Skor Kelengkapan (completeness_pct) ===
Rata-rata        : 99.8%
Minimum          : 97.5%
Maksimum         : 100.0%

✅ Semua dokumen memenuhi syarat kelengkapan ≥80% sesuai ketentuan tugas.


## 7. Simpan Output

Sesuai ketentuan tugas, hasil disimpan dengan penamaan berurutan `case_001.txt`, `case_002.txt`, dst,
ke folder `data/raw/`. Mapping nama file asli ↔ case_id disimpan agar dapat ditelusuri kembali pada Tahap 2
(Case Representation), karena nomor perkara asli ada di nama file PDF.


In [63]:
file_mapping = []

for idx, f in enumerate(pdf_files, start=1):
    case_id = f"case_{idx:03d}"
    out_path = RAW_DIR / f"{case_id}.txt"

    text_to_save = cleaned_texts[f.name]
    out_path.write_text(text_to_save, encoding="utf-8")

    file_mapping.append({
        "case_id": case_id,
        "original_filename": f.name,
        "extraction_method": raw_extracted[f.name][1],
        "char_count": len(text_to_save),
    })

    logger.info(f"Simpan {case_id} <- {f.name} ({len(text_to_save)} karakter)")

mapping_df = pd.DataFrame(file_mapping)
mapping_path = RAW_DIR / "file_mapping.csv"
mapping_df.to_csv(mapping_path, index=False)

print(f"✅ {len(file_mapping)} file teks disimpan ke: {RAW_DIR}")
print(f"✅ Mapping case_id <-> nama file asli disimpan ke: {mapping_path}")
mapping_df.head(10)


INFO | Simpan case_001 <- putusan_1002_pid.sus_2021_pn_jkt.tim_20260620230052.pdf (146723 karakter)
INFO | Simpan case_002 <- putusan_1037_pid.sus_2021_pn_jkt.tim_20260620225944.pdf (35683 karakter)
INFO | Simpan case_003 <- putusan_11_pid.sus_2022_pn_jkt.tim_20260620225936.pdf (104992 karakter)
INFO | Simpan case_004 <- putusan_202_pid.sus_2023_pn_jkt.tim_20260619115103.pdf (497686 karakter)
INFO | Simpan case_005 <- putusan_203_pid.sus_2023_pn_jkt.tim_20260620225510.pdf (514618 karakter)
INFO | Simpan case_006 <- putusan_212_pid.sus_2025_pn_jkt.tim_20260620224313.pdf (83039 karakter)
INFO | Simpan case_007 <- putusan_330_pid.sus_2025_pn_jkt.tim_20260620224225.pdf (56476 karakter)
INFO | Simpan case_008 <- putusan_351_pid.sus_2025_pn_jkt.tim_20260620223604.pdf (198620 karakter)
INFO | Simpan case_009 <- putusan_365_pid.sus_2023_pn_jkt.tim_20260620225531.pdf (166510 karakter)
INFO | Simpan case_010 <- putusan_463_pid.sus_2024_pn_jkt.tim_20260620225349.pdf (351847 karakter)
INFO | Simpa

✅ 30 file teks disimpan ke: C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\raw
✅ Mapping case_id <-> nama file asli disimpan ke: C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\data\raw\file_mapping.csv


,case_id,original_filename,extraction_method,char_count
0,case_001,putusan_1002_pid.sus_2021_pn_jkt.tim_202606202...,pymupdf,146723
1,case_002,putusan_1037_pid.sus_2021_pn_jkt.tim_202606202...,pymupdf,35683
2,case_003,putusan_11_pid.sus_2022_pn_jkt.tim_20260620225...,pymupdf,104992
3,case_004,putusan_202_pid.sus_2023_pn_jkt.tim_2026061911...,pymupdf,497686
4,case_005,putusan_203_pid.sus_2023_pn_jkt.tim_2026062022...,pymupdf,514618
5,case_006,putusan_212_pid.sus_2025_pn_jkt.tim_2026062022...,pymupdf,83039
6,case_007,putusan_330_pid.sus_2025_pn_jkt.tim_2026062022...,pymupdf,56476
7,case_008,putusan_351_pid.sus_2025_pn_jkt.tim_2026062022...,pymupdf,198620
8,case_009,putusan_365_pid.sus_2023_pn_jkt.tim_2026062022...,pymupdf,166510
9,case_010,putusan_463_pid.sus_2024_pn_jkt.tim_2026062022...,ocr,351847


In [64]:
logger.info(f"=== Selesai. Total dokumen diproses: {len(pdf_files)} ===")
logger.info(f"OK: {n_ok} | Perlu cek manual: {n_warn} | Gagal: {n_fail}")

print(f"\n📄 Log lengkap tersimpan di: {log_path}")
print("\n=== RINGKASAN AKHIR TAHAP 1 ===")
print(f"Total dokumen target     : ≥30")
print(f"Total dokumen tersedia   : {len(pdf_files)}")
print(f"Berhasil diproses (OK)   : {n_ok}")
print(f"Perlu review manual      : {n_warn}")
print(f"Gagal diproses           : {n_fail}")


INFO | === Selesai. Total dokumen diproses: 30 ===
INFO | OK: 30 | Perlu cek manual: 0 | Gagal: 0



📄 Log lengkap tersimpan di: C:\Users\maull\Downloads\Kuliah\SMT 6\Penalaran Komputer\New folder\project_cbr_ite\logs\cleaning.log

=== RINGKASAN AKHIR TAHAP 1 ===
Total dokumen target     : ≥30
Total dokumen tersedia   : 30
Berhasil diproses (OK)   : 30
Perlu review manual      : 0
Gagal diproses           : 0


## 8. Catatan
- Pola regex boilerplate (`WATERMARK_FRAGMENTS_RE`, `HEADER_NOMOR_RE`, dst.)
- Output Tahap 1 yang dihasilkan:
  - `data/raw/case_001.txt` ... `case_030.txt`
  - `data/raw/file_mapping.csv` (mapping case_id ↔ nama file asli, dibutuhkan di Tahap 2)
  - `logs/cleaning.log`

